# 05 — Presença em sessões e participação em votações

São medidas diferentes. Arquivos anuais são limitados a 80 MB. A ausência de voto individual não significa falta à sessão. O notebook não calcula taxa enquanto elegibilidade ou códigos forem incertos.

In [1]:
from pathlib import Path
import sys
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'validation_utils.py').exists())
sys.path.insert(0, str(root / 'scripts'))
from validation_utils import (Probe, OUT, CHECKS, MANIFEST, MANIFEST_FIELDS, api_camara, api_senado,
                              record, manifest_rows, save_manifest, rows_camara, count_values, csv_head)
pd.set_option('display.max_colwidth', 90)

import csv, io
from collections import Counter
probe=Probe('05',max_calls=100)
assert manifest_rows(), 'Execute 03 primeiro.'

## Câmara: evento concluído, presença em API e arquivo

In [2]:
event_id='82870'
event=probe.json(api_camara(f'eventos/{event_id}'))
att=probe.json(api_camara(f'eventos/{event_id}/deputados'))
url='https://dadosabertos.camara.leg.br/arquivos/eventosPresencaDeputados/csv/eventosPresencaDeputados-2026.csv'
bulk=probe.fetch(url)
api_ids={str(x.get('id')) for x in rows_camara(att.data)} if att.ok else set()
head=csv_head(bulk.content) if bulk.ok else []
columns=list(head[0]) if head else []
matches=[]
if bulk.ok:
    text=bulk.content.decode('utf-8-sig',errors='replace')
    delimiter=';' if text[:4096].count(';')>text[:4096].count(',') else ','
    reader=csv.DictReader(io.StringIO(text),delimiter=delimiter)
    event_col=next((c for c in reader.fieldnames if c.lower() in ('idevento','evento_id')),None)
    if event_col: matches=[r for r in reader if r.get(event_col)==event_id]
file_ids={str(r.get('idDeputado')) for r in matches if r.get('idDeputado')}
display(pd.DataFrame([{'evento':event_id,'situacao':(event.data or {}).get('dados',{}).get('situacao') if event.ok else None,
                      'API_presentes':len(api_ids),'arquivo_linhas_evento':len(matches),
                      'IDs_somente_API':len(api_ids-file_ids),'IDs_somente_arquivo':len(file_ids-api_ids),
                      'arquivo_colunas':columns}]))
record('05','presenca_camara','camara',url,len(matches),'Mesmo evento concluído em API e arquivo; denominador requer elegibilidade',
       {'API_presentes':len(api_ids),'arquivo_linhas_evento':len(matches),'IDs_somente_API':len(api_ids-file_ids),
        'IDs_somente_arquivo':len(file_ids-api_ids),'colunas':columns},
       'INCONCLUSIVO','Comparar IDs e regras de tipos de sessão, mandato e licença antes de taxa; falha: '+(bulk.error or att.error))

,evento,situacao,API_presentes,arquivo_linhas_evento,IDs_somente_API,IDs_somente_arquivo,arquivo_colunas
0,82870,Encerrada (Final),16,16,0,0,"[idEvento, uriEvento, dataHoraInicio, idDeputado, uriDeputado]"


{'notebook': '05',
 'check': 'presenca_camara',
 'fonte': 'camara',
 'url_parametros': 'https://dadosabertos.camara.leg.br/arquivos/eventosPresencaDeputados/csv/eventosPresencaDeputados-2026.csv',
 'coletado_em_utc': '2026-10-09T14:22:41+00:00',
 'amostra_n': 16,
 'regra': 'Mesmo evento concluído em API e arquivo; denominador requer elegibilidade',
 'contagens': '{"API_presentes": 16, "IDs_somente_API": 0, "IDs_somente_arquivo": 0, "arquivo_linhas_evento": 16, "colunas": ["idEvento", "uriEvento", "dataHoraInicio", "idDeputado", "uriDeputado"]}',
 'estado': 'INCONCLUSIVO',
 'motivo': 'Comparar IDs e regras de tipos de sessão, mandato e licença antes de taxa; falha: '}

## Senado: localizar fonte oficial de presença antes de qualquer taxa

In [3]:
catalog=list(csv.DictReader((root/'docs/catalogo_recursos.csv').open(encoding='utf-8-sig',newline='')))
import re
possible=[r for r in catalog if r['source']=='senado' and re.search(r'presen[çc]a|frequ[êe]ncia|comparecimento',
          (r['name']+' '+r['route_or_url']).lower())]
display(pd.DataFrame([{'tipo':r['resource_type'],'nome':r['name'],'url':r['route_or_url']} for r in possible]).head(20))
record('05','presenca_senado','senado','catálogo local e OpenAPI oficial',len(possible),
       'Fonte de presença e denominador de elegibilidade documentados',{'candidatos_catalogo':len(possible),'fonte_validada':0},
       'INCONCLUSIVO','Candidatos identificados são tipos de comparecimento em votação, não registros de presença em sessão; taxa bloqueada')

,tipo,nome,url
0,api_endpoint,Lista os Tipos de Comparecimento em Votações do Plenário,https://legis.senado.leg.br/dadosabertos/plenario/lista/tiposComparecimento
1,bulk_file_or_feed,ListaTiposComparecimento.csv,https://legis.senado.leg.br/dadosabertos/arquivos/ListaTiposComparecimento.csv
2,bulk_file_or_feed,ListaTiposComparecimento.json,https://legis.senado.leg.br/dadosabertos/arquivos/ListaTiposComparecimento.json
3,bulk_file_or_feed,ListaTiposComparecimento.xml,https://legis.senado.leg.br/dadosabertos/dados/ListaTiposComparecimento.xml


{'notebook': '05',
 'check': 'presenca_senado',
 'fonte': 'senado',
 'url_parametros': 'catálogo local e OpenAPI oficial',
 'coletado_em_utc': '2026-10-09T14:22:41+00:00',
 'amostra_n': 4,
 'regra': 'Fonte de presença e denominador de elegibilidade documentados',
 'contagens': '{"candidatos_catalogo": 4, "fonte_validada": 0}',
 'estado': 'INCONCLUSIVO',
 'motivo': 'Candidatos identificados são tipos de comparecimento em votação, não registros de presença em sessão; taxa bloqueada'}

## Votos: cobertura individual, arquivo e códigos brutos

In [4]:
votes_list=probe.json(api_camara('votacoes'),{'dataInicio':'2025-10-21','dataFim':'2025-10-22','itens':8})
ids=[x.get('id') for x in rows_camara(votes_list.data) if x.get('id')] if votes_list.ok else []
detail=[]
for vid in ids[:6]:
    d=probe.json(api_camara(f'votacoes/{vid}'))
    v=probe.json(api_camara(f'votacoes/{vid}/votos'))
    detail.append({'id':vid,'aprovacao':(d.data or {}).get('dados',{}).get('aprovacao') if d.ok else None,
                   'votos_individuais':len(rows_camara(v.data)) if v.ok else None,'erro':v.error})
display(pd.DataFrame(detail))
url='https://dadosabertos.camara.leg.br/arquivos/votacoesVotos/csv/votacoesVotos-2025.csv'
bulk=probe.fetch(url)
head=csv_head(bulk.content) if bulk.ok else []
columns=list(head[0]) if head else []
target_id=head[0].get('idVotacao') if head else None
file_votes=0
if bulk.ok and target_id:
    text=bulk.content.decode('utf-8-sig',errors='replace')
    delimiter=';' if text[:4096].count(';')>text[:4096].count(',') else ','
    file_votes=sum(row.get('idVotacao')==target_id for row in csv.DictReader(io.StringIO(text),delimiter=delimiter))
target_api=probe.json(api_camara(f'votacoes/{target_id}/votos')) if target_id else None
target_api_votes=len(rows_camara(target_api.data)) if target_api and target_api.ok else None
display(pd.DataFrame([{'votacao_positiva_arquivo':target_id,'votos_arquivo':file_votes,'votos_API':target_api_votes}]))
record('05','votos_camara','camara',url,len(detail),'Identificar votações com votos individuais; arquivo anual e endpoint reconciliáveis',
       {'votacoes_amostradas':len(detail),'com_votos':sum((x['votos_individuais'] or 0)>0 for x in detail),
        'exemplo_positivo_id':target_id,'votos_arquivo_exemplo':file_votes,'votos_API_exemplo':target_api_votes,'colunas_arquivo':columns},
       'INCONCLUSIVO','Voto ausente no endpoint não identifica abstenção; nominalidade e cobertura a validar. '+bulk.error)
sen=probe.json(api_senado('votacao.json'),{'dataInicio':'2025-10-21','dataFim':'2025-10-21'})
records=sen.data if sen.ok and isinstance(sen.data,list) else []
codes=Counter(v.get('siglaVotoParlamentar') for x in records for v in x.get('votos',[]))
display(pd.DataFrame([{'codigo_bruto':k,'n':v} for k,v in codes.items()]))
record('05','codigos_voto_senado','senado',sen.url,len(records),'Preservar todos os códigos; mapear apenas após dicionário oficial',
       {'votacoes':len(records),'codigos':dict(codes)},'INCONCLUSIVO',
       'Votou e demais códigos ainda não determinam direção ou ausência; '+sen.error)

,id,aprovacao,votos_individuais,erro
0,2536009-6,1,0,
1,2537826-23,1,0,
2,2456284-57,1,0,
3,2456284-55,0,0,
4,2456284-51,1,0,
5,2558775-35,1,0,


,votacao_positiva_arquivo,votos_arquivo,votos_API
0,106701-223,429,429


,codigo_bruto,n
0,Não,18
1,Sim,36
2,AP,10
3,MIS,4
4,Presidente (art. 51 RISF),1
5,P-NRV,9
6,LS,2
7,NCom,1


{'notebook': '05',
 'check': 'codigos_voto_senado',
 'fonte': 'senado',
 'url_parametros': 'https://legis.senado.leg.br/dadosabertos/votacao.json?dataInicio=2025-10-21&dataFim=2025-10-21',
 'coletado_em_utc': '2026-10-09T14:22:58+00:00',
 'amostra_n': 1,
 'regra': 'Preservar todos os códigos; mapear apenas após dicionário oficial',
 'contagens': '{"codigos": {"AP": 10, "LS": 2, "MIS": 4, "NCom": 1, "Não": 18, "P-NRV": 9, "Presidente (art. 51 RISF)": 1, "Sim": 36}, "votacoes": 1}',
 'estado': 'INCONCLUSIVO',
 'motivo': 'Votou e demais códigos ainda não determinam direção ou ausência; '}